# 03 — Preprocessing Mathematics

Learn how descriptive statistics and scaling relate to ML inputs using the uploaded dataset.

In [7]:
from pathlib import Path
import sys
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from src.data_loading import load_uploaded
from src.preprocessing import descriptive_statistics, split_aware_standardise
df = load_uploaded()
display(descriptive_statistics(df))

,feature,count,mean,std,min,5%,25%,50%,75%,95%,max,missing,variance,skewness,iqr
0,Age,20000.0,49.565400,17.320956,20.0,22.000000,35.000000,50.00000,64.000000,77.00000,79.0,0,300.015524,-0.001846,29.000000
1,Glucose,20000.0,119.603400,29.590699,40.0,71.000000,99.000000,120.00000,140.000000,169.00000,200.0,0,875.609489,0.001392,41.000000
2,Blood Pressure,20000.0,69.432850,12.016115,30.0,50.000000,61.000000,69.00000,78.000000,89.00000,114.0,0,144.387010,-0.011413,17.000000
3,Skin Thickness,20000.0,20.105450,9.130969,7.0,7.000000,13.000000,20.00000,26.000000,36.00000,57.0,0,83.374599,0.402495,13.000000
4,Insulin,20000.0,83.952550,53.026841,15.0,15.000000,40.000000,79.00000,120.000000,180.00000,276.0,0,2811.845891,0.534908,80.000000
5,BMI,20000.0,32.008980,6.913271,15.0,20.449993,27.314424,31.99404,36.750196,43.45665,50.0,0,47.793318,0.018217,9.435772
6,Cholesterol,20000.0,189.555400,35.081701,100.0,132.000000,166.000000,190.00000,213.000000,248.00000,320.0,0,1230.725767,0.029942,47.000000
7,Diabetes Pedigree Function,20000.0,1.310665,0.693984,0.1,0.229950,0.704750,1.31600,1.914000,2.38700,2.5,0,0.481613,-0.013841,1.209250


## Mean, median, variance, standard deviation and IQR

Mean is the arithmetic average; median is the midpoint; variance measures squared spread; standard deviation is the square root of variance; IQR = Q3 − Q1 and is less sensitive to extremes.

In [8]:
numeric = df.select_dtypes(include=np.number)
display(numeric.agg(["mean", "median", "var", "std", "min", "max"]).T)

,mean,median,var,std,min,max
Age,49.565400,50.00000,300.015524,17.320956,20.0,79.0
Glucose,119.603400,120.00000,875.609489,29.590699,40.0,200.0
Blood Pressure,69.432850,69.00000,144.387010,12.016115,30.0,114.0
Skin Thickness,20.105450,20.00000,83.374599,9.130969,7.0,57.0
Insulin,83.952550,79.00000,2811.845891,53.026841,15.0,276.0
BMI,32.008980,31.99404,47.793318,6.913271,15.0,50.0
Cholesterol,189.555400,190.00000,1230.725767,35.081701,100.0,320.0
Diabetes Pedigree Function,1.310665,1.31600,0.481613,0.693984,0.1,2.5


## Standardisation and data leakage

For feature value x, training mean μ and training standard deviation σ: z = (x − μ) / σ. Split before fitting a scaler; fitting it on all rows leaks information about the held-out test set. Scaling is often useful for Logistic Regression, while tree models usually do not require it.

In [9]:
target = "Outcome"
X = df.select_dtypes(include=np.number).drop(columns=[target], errors="ignore")
y = df[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2026, stratify=y)
X_train_scaled, X_test_scaled, scaler = split_aware_standardise(X_train, X_test)
summary = pd.DataFrame({"mean_before": X_train.mean(), "std_before": X_train.std(ddof=0), "mean_after": X_train_scaled.mean(), "std_after": X_train_scaled.std(ddof=0)})
display(summary.round(4))
print("Train/test rows:", len(X_train), len(X_test))

,mean_before,std_before,mean_after,std_after
Age,49.7236,17.3376,0.0,1.0
Glucose,119.7004,29.6140,-0.0,1.0
Blood Pressure,69.4992,11.9780,-0.0,1.0
Skin Thickness,20.1904,9.1647,0.0,1.0
Insulin,84.0353,53.2441,-0.0,1.0
BMI,31.9978,6.9409,-0.0,1.0
Cholesterol,189.5411,35.0704,0.0,1.0
Diabetes Pedigree Function,1.3057,0.6935,-0.0,1.0


Train/test rows: 16000 4000


## Correlation caveats

Pearson correlation measures linear association from −1 to +1. It can miss nonlinear relationships, be affected by outliers and be misleading for arbitrary category codes. Correlation does not establish causation. Use pipelines to fit imputers, scalers and feature selectors inside training folds.